In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
import catboost as cb
import xgboost as xgb


In [2]:
warnings.filterwarnings('ignore')

SEED = 63
SEEDS = [63, 163, 263, 363, 463]
SHIFT = 42 

try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except ImportError:
    HAS_GPU = False

CB_TASK = 'GPU' if HAS_GPU else 'CPU'
XGB_DEV = 'cuda' if HAS_GPU else 'cpu'


In [3]:
# Load Data
DATA_DIR = '/kaggle/input/competitions/ch-27-celebal-technologies-jadavour-university'
if not os.path.exists(DATA_DIR):
    DATA_DIR = './'

tr_df = pd.read_csv(os.path.join(DATA_DIR, 'orders_train.csv'))
te_df = pd.read_csv(os.path.join(DATA_DIR, 'orders_test.csv'))
meta = pd.read_csv(os.path.join(DATA_DIR, 'hub_metadata.csv'))

tr_df['Date'] = pd.to_datetime(tr_df['Date'])
te_df['Date'] = pd.to_datetime(te_df['Date'])
print(f"Train shape: {tr_df.shape} | Test shape: {te_df.shape}")

# Clean Metadata
meta['CompetitorDistance'] = meta['CompetitorDistance'].fillna(meta['CompetitorDistance'].median())
meta['CompetitorOpenSinceMonth'] = meta['CompetitorOpenSinceMonth'].fillna(0).astype(int)
meta['CompetitorOpenSinceYear'] = meta['CompetitorOpenSinceYear'].fillna(0).astype(int)
meta['LoyaltyProgramSinceYear'] = meta['LoyaltyProgramSinceYear'].fillna(0).astype(int)
meta['LoyaltyProgramSinceWeek'] = meta['LoyaltyProgramSinceWeek'].fillna(0).astype(int)
meta['LoyaltyProgramInterval'] = meta['LoyaltyProgramInterval'].fillna('None')

tr_df = tr_df.merge(meta, on='HubID', how='left')
te_df = te_df.merge(meta, on='HubID', how='left')


Train shape: (970379, 9) | Test shape: (46830, 8)


In [4]:
# Feature Engineering
int_mos = {
    'Jan,Apr,Jul,Oct':  {1, 4, 7, 10},
    'Feb,May,Aug,Nov':  {2, 5, 8, 11},
    'Mar,Jun,Sept,Dec': {3, 6, 9, 12},
}

for df in [tr_df, te_df]:
    df.sort_values(['HubID', 'Date'], inplace=True)
    df.reset_index(drop=True, inplace=True)

    df['Year'] = df['Date'].dt.year
    df['Month'] = df['Date'].dt.month
    df['DayOfWeek'] = df['Date'].dt.dayofweek
    df['DayOfYear'] = df['Date'].dt.dayofyear
    df['sin_dow'] = np.sin(2 * np.pi * df['DayOfWeek'] / 7)
    df['cos_dow'] = np.cos(2 * np.pi * df['DayOfWeek'] / 7)
    df['sin_doy'] = np.sin(2 * np.pi * df['DayOfYear'] / 365.25)
    df['cos_doy'] = np.cos(2 * np.pi * df['DayOfYear'] / 365.25)

    df['LogCompDist'] = np.log1p(df['CompetitorDistance'])
    df['CompOpenMonths'] = (12 * (df['Year'] - df['CompetitorOpenSinceYear']) + (df['Month'] - df['CompetitorOpenSinceMonth'])).clip(lower=0)
    df.loc[df['CompetitorOpenSinceYear'] == 0, 'CompOpenMonths'] = 0
    df['CompIsOpen'] = (df['CompOpenMonths'] > 0).astype(int)

    df['IsLoyaltyActive'] = 0
    for interval, months in int_mos.items():
        mask = (
            (df['LoyaltyProgram'] == 1) &
            (df['LoyaltyProgramInterval'] == interval) &
            (df['Month'].isin(months)) &
            (df['Year'] >= df['LoyaltyProgramSinceYear'])
        )
        df.loc[mask, 'IsLoyaltyActive'] = 1

    df['PromoPrev'] = df.groupby('HubID')['PromoActive'].shift(1).fillna(0).astype(int)
    df['PromoNext'] = df.groupby('HubID')['PromoActive'].shift(-1).fillna(0).astype(int)
    df['Promo_DoW'] = df['PromoActive'] * 10 + df['DayOfWeek']

print("Base features built.")

# Lag Features
tr_df['is_test'] = 0
te_df['is_test'] = 1
te_df['OrderVolume'] = np.nan

full = pd.concat([tr_df, te_df], ignore_index=True).sort_values(['HubID', 'Date']).reset_index(drop=True)
full['LogVol'] = np.log1p(full['OrderVolume'])

for s in [42, 49, 56, 63]:
    full[f'Lag_{s}'] = full.groupby('HubID')['LogVol'].shift(s)

full['RollMean14_L42'] = full.groupby('HubID')['Lag_42'].transform(lambda x: x.rolling(14, min_periods=1).mean())
full['RollMean28_L42'] = full.groupby('HubID')['Lag_42'].transform(lambda x: x.rolling(28, min_periods=1).mean())
full['LagDiff_42_49'] = full['Lag_42'] - full['Lag_49']

tr_df = full[full['is_test'] == 0].copy()
te_df = full[full['is_test'] == 1].copy()
del full

lag_cols = ['Lag_42', 'Lag_49', 'Lag_56', 'Lag_63', 'RollMean14_L42', 'RollMean28_L42', 'LagDiff_42_49']
print("Lag features built.")

# Target Encoding Helper
def add_hub_encodings(src_df, tgt_df, g_mean, k=30):
    src = src_df[(src_df['IsOpen'] == 1) & (src_df['OrderVolume'] > 0)]
    
    h = src.groupby('HubID')['LogVol'].agg(['mean', 'count'])
    a_hub = h['count'] / (h['count'] + k)
    hub_map = (a_hub * h['mean'] + (1 - a_hub) * g_mean).to_dict()

    hd = src.groupby(['HubID', 'DayOfWeek'])['LogVol'].agg(['mean', 'count'])
    a_dow = hd['count'] / (hd['count'] + k)
    dow_map = (a_dow * hd['mean'] + (1 - a_dow) * g_mean).to_dict()

    out = tgt_df.copy()
    out['HubMeanLV'] = out['HubID'].map(hub_map).fillna(g_mean)
    keys = list(zip(out['HubID'], out['DayOfWeek']))
    out['HubDoWMeanLV'] = [dow_map.get(k, g_mean) for k in keys]
    return out

g_mean = tr_df[(tr_df['IsOpen'] == 1) & (tr_df['OrderVolume'] > 0)]['LogVol'].mean()

features = [
    'HubID', 'HubFormat', 'AssortmentTier', 'CompetitorDistance', 'LogCompDist', 
    'CompOpenMonths', 'CompIsOpen', 'LoyaltyProgram', 'IsLoyaltyActive',
    'DayOfWeek', 'Month', 'Year', 'DayOfYear', 'sin_dow', 'cos_dow', 'sin_doy', 'cos_doy',
    'PromoActive', 'PromoPrev', 'PromoNext', 'Promo_DoW', 'RegionalHoliday', 'SchoolClosureFlag',
    *lag_cols, 'HubMeanLV', 'HubDoWMeanLV'
]
cat_cols = ['HubFormat', 'AssortmentTier', 'DayOfWeek', 'Month', 'Promo_DoW']
print(f"Selected {len(features)} features.")

def rmsle(y_true, y_pred):
    return np.sqrt(np.mean((np.log1p(np.clip(y_pred, 0, None)) - np.log1p(y_true)) ** 2))

# Cross Validation
print("\nStarting 3-Fold Time-Series CV")
max_date = tr_df['Date'].max()
folds = []
for i in range(3):
    ve = max_date - pd.Timedelta(days=SHIFT * i)
    vs = ve - pd.Timedelta(days=SHIFT - 1)
    te = vs - pd.Timedelta(days=1)
    folds.append((te, vs, ve))
folds.reverse()

mask_ok = (tr_df['IsOpen'] == 1) & (tr_df['OrderVolume'] > 0)
cv_scores = {'LGB': [], 'CB': [], 'XGB': [], 'AVG': []}
best_iters = {'LGB': [], 'CB': [], 'XGB': []}

for fi, (tr_end, vs, ve) in enumerate(folds):
    print(f"\nFold {fi+1}: Train <= {tr_end.date()} | Val {vs.date()} to {ve.date()}")

    tr_m = (tr_df['Date'] <= tr_end) & mask_ok & tr_df['Lag_42'].notna()
    va_m = (tr_df['Date'] >= vs) & (tr_df['Date'] <= ve) & mask_ok & tr_df['Lag_42'].notna()

    tr, va = tr_df[tr_m].copy(), tr_df[va_m].copy()
    tr = add_hub_encodings(tr, tr, g_mean)
    va = add_hub_encodings(tr, va, g_mean)

    for c in lag_cols:
        tr[c] = tr[c].fillna(g_mean)
        va[c] = va[c].fillna(g_mean)

    X_tr, y_tr = tr[features], tr['LogVol']
    X_va, y_va = va[features], va['LogVol']
    y_true = va['OrderVolume'].values

    # LightGBM
    Xt, Xv = X_tr.copy(), X_va.copy()
    for c in cat_cols:
        Xt[c], Xv[c] = Xt[c].astype('category'), Xv[c].astype('category')

    dt = lgb.Dataset(Xt, y_tr, categorical_feature=cat_cols)
    dv = lgb.Dataset(Xv, y_va, categorical_feature=cat_cols, reference=dt)
    m_lgb = lgb.train(
        {'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt',
         'learning_rate': 0.02, 'num_leaves': 127, 'max_depth': 10,
         'feature_fraction': 0.7, 'bagging_fraction': 0.8, 'bagging_freq': 1,
         'min_child_samples': 20, 'reg_alpha': 0.1, 'reg_lambda': 1.0,
         'verbose': -1, 'n_jobs': -1, 'random_state': SEED},
        dt, 3000, valid_sets=[dv], callbacks=[lgb.early_stopping(200, verbose=False)]
    )
    p_lgb = np.expm1(m_lgb.predict(Xv))
    best_iters['LGB'].append(m_lgb.best_iteration)
    score_lgb = rmsle(y_true, p_lgb)
    cv_scores['LGB'].append(score_lgb)
    print(f"LGB : {score_lgb:.5f} (iter {m_lgb.best_iteration})")

    # CatBoost
    Xt, Xv = X_tr.copy(), X_va.copy()
    for c in cat_cols:
        Xt[c], Xv[c] = Xt[c].astype(str), Xv[c].astype(str)

    m_cb = cb.CatBoostRegressor(
        iterations=3000, learning_rate=0.02, depth=8, l2_leaf_reg=3.0, 
        random_strength=0.5, task_type=CB_TASK, verbose=0, random_seed=SEED
    )
    m_cb.fit(Xt, y_tr, cat_features=cat_cols, eval_set=(Xv, y_va), early_stopping_rounds=200, verbose=0)
    p_cb = np.expm1(m_cb.predict(Xv))
    best_iters['CB'].append(m_cb.best_iteration_)
    score_cb = rmsle(y_true, p_cb)
    cv_scores['CB'].append(score_cb)
    print(f"CB  : {score_cb:.5f} (iter {m_cb.best_iteration_})")

    # XGBoost
    Xt, Xv = X_tr.copy(), X_va.copy()
    for c in cat_cols:
        Xt[c], Xv[c] = Xt[c].astype('category'), Xv[c].astype('category')

    m_xgb = xgb.XGBRegressor(
        n_estimators=3000, learning_rate=0.02, max_depth=9, colsample_bytree=0.7, 
        subsample=0.8, reg_alpha=0.1, reg_lambda=1.0, tree_method='hist', 
        device=XGB_DEV, enable_categorical=True, verbosity=0, random_state=SEED, 
        early_stopping_rounds=200
    )
    m_xgb.fit(Xt, y_tr, eval_set=[(Xv, y_va)], verbose=0)
    p_xgb = np.expm1(m_xgb.predict(Xv))
    best_iters['XGB'].append(m_xgb.best_iteration)
    score_xgb = rmsle(y_true, p_xgb)
    cv_scores['XGB'].append(score_xgb)
    print(f"XGB : {score_xgb:.5f} (iter {m_xgb.best_iteration})")

    p_avg = (p_lgb + p_cb + p_xgb) / 3.0
    score_avg = rmsle(y_true, p_avg)
    cv_scores['AVG'].append(score_avg)
    print(f"AVG : {score_avg:.5f}")

print("\nCV Summary (Mean +/- Std):")
for name, scores in cv_scores.items():
    print(f"{name:4s} RMSLE = {np.mean(scores):.5f} +/- {np.std(scores):.5f}")

final_rounds = {}
for name, iters in best_iters.items():
    final_rounds[name] = min(int(np.median(iters) * 1.05), 3000)
    print(f"{name} final rounds: {final_rounds[name]} (median CV: {int(np.median(iters))})")

# Full Training
print(f"\nFull Training: {len(SEEDS)} seeds x 3 models")
tr_df = add_hub_encodings(tr_df, tr_df, g_mean)
te_df = add_hub_encodings(tr_df, te_df, g_mean)

for c in lag_cols:
    tr_df[c] = tr_df[c].fillna(g_mean)
    te_df[c] = te_df[c].fillna(g_mean)

fm = (tr_df['IsOpen'] == 1) & (tr_df['OrderVolume'] > 0) & tr_df['Lag_42'].notna()
X_all = tr_df[fm][features].copy()
y_all = tr_df[fm]['LogVol']
X_tst = te_df[features].copy()
N = len(SEEDS)

# LightGBM Seeds
X_al, X_tl = X_all.copy(), X_tst.copy()
for c in cat_cols:
    X_al[c], X_tl[c] = X_al[c].astype('category'), X_tl[c].astype('category')

print("Training LGB...", end=" ")
preds_lgb = np.zeros(len(X_tst))
for s in SEEDS:
    d = lgb.Dataset(X_al, y_all, categorical_feature=cat_cols)
    m = lgb.train(
        {'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt',
         'learning_rate': 0.02, 'num_leaves': 127, 'max_depth': 10, 'feature_fraction': 0.7, 
         'bagging_fraction': 0.8, 'bagging_freq': 1, 'min_child_samples': 20, 
         'reg_alpha': 0.1, 'reg_lambda': 1.0, 'verbose': -1, 'n_jobs': -1, 'random_state': s},
        d, final_rounds['LGB']
    )
    preds_lgb += np.expm1(m.predict(X_tl)) / N
    print(f"{s} ", end="", flush=True)
print()

# CatBoost Seeds
X_ac, X_tc = X_all.copy(), X_tst.copy()
for c in cat_cols:
    X_ac[c], X_tc[c] = X_ac[c].astype(str), X_tc[c].astype(str)

print("Training CB...", end=" ")
preds_cb = np.zeros(len(X_tst))
for s in SEEDS:
    m = cb.CatBoostRegressor(
        iterations=final_rounds['CB'], learning_rate=0.02, depth=8, l2_leaf_reg=3.0, 
        random_strength=0.5, task_type=CB_TASK, verbose=0, random_seed=s
    )
    m.fit(X_ac, y_all, cat_features=cat_cols)
    preds_cb += np.expm1(m.predict(X_tc)) / N
    print(f"{s} ", end="", flush=True)
print()

# XGBoost Seeds
X_ax, X_tx = X_all.copy(), X_tst.copy()
for c in cat_cols:
    X_ax[c], X_tx[c] = X_ax[c].astype('category'), X_tx[c].astype('category')

print("Training XGB...", end=" ")
preds_xgb = np.zeros(len(X_tst))
for s in SEEDS:
    m = xgb.XGBRegressor(
        n_estimators=final_rounds['XGB'], learning_rate=0.02, max_depth=9, colsample_bytree=0.7, 
        subsample=0.8, reg_alpha=0.1, reg_lambda=1.0, tree_method='hist', device=XGB_DEV, 
        enable_categorical=True, verbosity=0, random_state=s
    )
    m.fit(X_ax, y_all)
    preds_xgb += np.expm1(m.predict(X_tx)) / N
    print(f"{s} ", end="", flush=True)
print()

# Ensemble & Formatting
final = (preds_lgb + preds_cb + preds_xgb) / 3.0
final[te_df['IsOpen'].values == 0] = 0.0
final = np.clip(final, 0, None)

sub = pd.DataFrame({'Id': te_df['Id'].values, 'OrderVolume': final})
sub.to_csv('submission.csv', index=False)
print(f"\nSaved submission.csv ({len(sub)} rows)")
print(sub.head(10).to_string())

Base features built.
Lag features built.
Selected 32 features.

Starting 3-Fold Time-Series CV

Fold 1: Train <= 2015-02-13 | Val 2015-02-14 to 2015-03-27
LGB : 0.12422 (iter 2965)
CB  : 0.12602 (iter 2913)
XGB : 0.12352 (iter 2998)
AVG : 0.12310

Fold 2: Train <= 2015-03-27 | Val 2015-03-28 to 2015-05-08
LGB : 0.15880 (iter 706)
CB  : 0.16280 (iter 1134)
XGB : 0.14620 (iter 548)
AVG : 0.15380

Fold 3: Train <= 2015-05-08 | Val 2015-05-09 to 2015-06-19
LGB : 0.11919 (iter 2079)
CB  : 0.12415 (iter 2992)
XGB : 0.11986 (iter 1514)
AVG : 0.11968

CV Summary (Mean +/- Std):
LGB  RMSLE = 0.13407 +/- 0.01761
CB   RMSLE = 0.13765 +/- 0.01780
XGB  RMSLE = 0.12986 +/- 0.01165
AVG  RMSLE = 0.13220 +/- 0.01534
LGB final rounds: 2182 (median CV: 2079)
CB final rounds: 3000 (median CV: 2913)
XGB final rounds: 1589 (median CV: 1514)

Full Training: 5 seeds x 3 models
Training LGB... 63 163 263 363 463 
Training CB... 63 163 263 363 463 
Training XGB... 63 163 263 363 463 

Saved submission.csv (4683

To predict daily OrderVolume across 1,115 hubs, the target variable was first transformed using log1p to stabilize its heavy right-skew, and closed hubs were isolated to be hardcoded to zero. Exploratory analysis highlighted strong day-of-week and promotional patterns, prompting the use of cyclical sine and cosine encodings for calendar features and specific interaction flags for previous and upcoming promotions. A critical metadata bug regarding loyalty program intervals for September was also identified and fixed to ensure accurate temporal mapping.

The feature engineering approach prioritized a concise set of 32 high-signal features over high dimensionality to prevent leaderboard regression. To strictly avoid target leakage, all lag features and rolling averages (like 14-day and 28-day trends) were shifted by a minimum of 42 days to align exactly with the inference horizon. Bayesian-smoothed target encodings were applied to categorical variables and computed strictly within training folds. To address a significant disconnect between local cross-validation and the leaderboard, the validation schema utilized a 3-fold expanding-window time-series split with exactly 42-day validation periods to mimic the test environment.

The final predictive architecture relies on a tri-model gradient-boosted ensemble consisting of LightGBM, CatBoost, and XGBoost to capture structural diversity. Rather than using a fixed iteration count, early stopping during cross-validation determined the optimal boost rounds, using the median best iteration across folds for the final full-dataset retrain. To maximize stability and cancel out individual architectural biases, each framework was trained across five different random seeds, and the final predictions were combined using a simple equal-weight average rather than optimized blend weights to prevent overfitting to specific temporal periods. Finally, predictions were inverted using expm1, negative values were clipped to zero, and closed-day predictions were safely zeroed out.
